# Sweep: vary a design parameter and tool settings

This notebook runs the SystemVerilog version of the Trivium stream cipher of the examples
through `yosys_fpga+nextpnr` for an ECP5 device. It changes one design parameter (`G_OUT_BITS`,
the keystream bits per clock), one setting of `yosys_fpga` (`nowidelut`) and one setting of
`nextpnr` (the seed). It collects the results in a table and plots the Pareto front of LUTs and
throughput. At the end, it reads the same results again from the run root.

The notebook needs Yosys, `nextpnr-ecp5`, pandas and Plotly.

In [ ]:
import os
from pathlib import Path

import xeda

EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(f"No examples in {EXAMPLES}. Set XEDA_EXAMPLES to the examples directory.")

WORK = Path("notebook-output").resolve()
ws = xeda.Workspace(WORK / "xeda_run", hashed_run_dirs=True, local_settings=xeda.local_settings())

## Build the specs

A sweep takes any iterable of `Spec`. You build the specs with plain Python.

`Design.with_updates` returns a new design and leaves the original as it is. It accepts a dotted
key. The sweep also varies one synthesis setting, `nowidelut`. By default, ABC9 can map logic to
wide LUTs of five to seven inputs, which ECP5 builds from LUT4s and multiplexer cells.
`nowidelut: true` maps to LUT4s only. Each configuration runs with three seeds of `nextpnr`,
because the seed alone can change `fmax` by ten percent or more.

All points have a clock period of 10 ns. The clock decides only whether a point passes:
`nextpnr-ecp5` places and routes the same way for every target, and Xeda gives ABC9 no target
from the clock. The last point makes 288 bits per clock. Its logic is too slow for this clock on
the device, so that point does not pass.

In [ ]:
from collections import Counter
from itertools import product

import pandas as pd
import plotly.graph_objects as go

from xeda import Design, Spec
from xeda.errors import RunReplacedError, SpecError

base = Design.load(EXAMPLES / "sv/trivium/trivium.yaml")


def point(bits: int, nowidelut: bool, seed: int) -> Spec:
    return Spec(
        "yosys_fpga+nextpnr",
        base.with_updates({"rtl.parameters.G_OUT_BITS": bits}),
        settings={
            "yosys_fpga": {
                "fpga.part": "LFE5U-85F-6BG381C",
                "clock.period": "10 ns",
                "nowidelut": nowidelut,
            },
            "nextpnr": {"lpf_allow_unconstrained": True, "seed": seed},
        },
    )


specs = [point(*p) for p in product((64, 128, 192), (False, True), (1, 2, 3))]
specs.append(point(288, False, 1))
print(len(specs), "points")

## Use hashed run directories

These specs differ in a design parameter. With the default layout, they would all use the run
directory `trivium/yosys_fpga`. Xeda refuses this before it runs anything.

In [ ]:
plain = xeda.Workspace(WORK / "plain", local_settings=xeda.local_settings())
try:
    plain.sweep(specs)
except SpecError as error:
    print(error.code)
    print(error.points)

A workspace with `hashed_run_dirs=True` gives each configuration its own directory. The first cell
of this notebook made `ws` that way.

## Run the sweep

Xeda plans all points before it runs one. `max_workers=os.cpu_count()` runs up to one node per
processor at a time, each in its own worker process. `os.cpu_count()` counts all processors of the
host; on a machine that gives you fewer, set `max_workers` to your share. More workers help until
every node that can run is running: here, seven syntheses at first. After that, the longest chain,
the 288-bit synthesis and its place and route, sets the time. The own threads of `nextpnr`
(`nthreads`) made little difference to this sweep.

In [ ]:
sweep = ws.sweep(specs, max_workers=os.cpu_count() or 1)
print({outcome.value: count for outcome, count in sweep.counts().items()})

Points that need the same node run it once. The seed does not change the synthesis, so the three
seeds of one width and one setting share one `yosys_fpga` run. The first point of the group
reports `ran`. The others report `shared`. The 288-bit point has one seed, so its synthesis serves
only that point.

In [ ]:
dispositions = Counter(p.result.nodes["yosys_fpga"].disposition.value for p in sweep)
print(dict(sorted(dispositions.items())))

## Make a table

`to_pandas` gives one row for each point, in input order. You choose the metrics. A metric is
named `<node>.<metric>`, for example `nextpnr.fmax`. The sweep keeps the points that did not pass.
A point has `None` for a metric that the record of its node does not hold.

In [ ]:
df = sweep.to_pandas(metrics=("nextpnr.fmax", "nextpnr.lut")).rename(
    columns={
        "design.rtl.parameters.G_OUT_BITS": "bits",
        "settings.yosys_fpga.nowidelut": "nowidelut",
        "settings.nextpnr.seed": "seed",
        "nextpnr.fmax": "fmax",
        "nextpnr.lut": "lut",
    }
)
columns = ["bits", "nowidelut", "seed", "outcome", "error_code", "fmax", "lut"]
df[columns].to_csv(WORK / "trivium-sweep.csv", index=False)
df[columns]

To see why a point did not pass, read its result.

In [ ]:
for p in sweep:
    if not p.result.passed:
        print(p.index, p.result.outcome.value, p.result.error.code)
        print(p.result.error.message.splitlines()[0])

## Plot the Pareto front

A wider datapath makes more keystream bits per clock, and it costs LUTs. The throughput of a
configuration is its bits per clock times its `fmax`. The seed changes `fmax` but not the netlist,
so the next cell takes the median of the three seeds, and the plot shows a bar from the lowest to
the highest seed. A configuration is on the Pareto front when no other configuration uses at most
as many LUTs and reaches at least as high a median throughput, and is better in one of the two.
The front is a staircase: it shows the highest throughput for each number of LUTs.

Trivium computes up to 64 bits in parallel, because each of them depends only on the state of the
clock before. So up to 64 bits, the throughput grows with the width. Beyond 64 bits, some bits
depend on bits of the same clock. The logic gets deeper, `fmax` drops, and each LUT buys less
throughput. Both axes are logarithmic.

`nowidelut` matters most at 192 bits. There, it maps to about a third of the LUTs, and its slowest
seed is faster than the fastest seed of the default. At 128 bits, it saves about a quarter of the
LUTs for about the same `fmax`. At 64 bits, it saves less than a tenth of the LUTs, and its `fmax`
differs from the default by less than the seeds do. So the effect of one setting depends on the size
of the design, and a sweep measures it with one more key. Xeda keeps wide LUTs by default, because
on other designs they are faster.

Compare configurations by their results after place and route, with more than one seed. The LUT
count of Yosys does not tell which netlist is faster. To tune `nextpnr` too, choose the synthesis
first, then sweep the options of `nextpnr` on that netlist. With Vivado, a sweep of the clock
target spreads the results, because its timing-driven steps work toward the target. With
`nextpnr-ecp5`, the clock period is no axis to sweep.

In [ ]:
passed = df[df.outcome == "passed"]
if passed.empty:
    raise RuntimeError("No point passed, so there is nothing to plot. See the failures above.")
first_passed = next(p for p in sweep if p.result.passed)
assert first_passed.result.nodes["nextpnr"].record is not None
unit = first_passed.result.nodes["nextpnr"].record.metric_info["fmax"].unit
assert unit == "MHz", unit  # bits per clock times MHz is Mbit/s, so `throughput` is in Gbit/s

# One row for each configuration: the seeds share the netlist, so they share the LUT count.
medians = (
    passed.groupby(["bits", "nowidelut"])
    .agg(lut=("lut", "first"), fmax=("fmax", "median"), low=("fmax", "min"), high=("fmax", "max"))
    .reset_index()
    .assign(throughput=lambda d: d.bits * d.fmax / 1000)
)
print(medians.round(1).to_string(index=False))

In [ ]:
ranked = medians.sort_values(["lut", "throughput"], ascending=[True, False])
front = ranked[ranked.throughput > ranked.throughput.cummax().shift(fill_value=0)]

fig = go.Figure()
for nowidelut, points in medians.groupby("nowidelut"):
    fig.add_scatter(
        x=points.lut,
        y=points.throughput,
        mode="markers",
        name=f"nowidelut: {nowidelut}",
        error_y={
            "type": "data",
            "symmetric": False,
            "array": points.bits * (points.high - points.fmax) / 1000,
            "arrayminus": points.bits * (points.fmax - points.low) / 1000,
        },
        text=[
            f"{b} bits, median {f:.1f} MHz (seeds: {lo:.1f} to {hi:.1f} MHz)"
            for b, f, lo, hi in zip(points.bits, points.fmax, points.low, points.high)
        ],
        hovertemplate="%{text}<br>%{x} LUTs, %{y:.2f} Gbit/s<extra></extra>",
    )
fig.add_scatter(
    x=front.lut,
    y=front.throughput,
    mode="lines+markers",
    name="Pareto front",
    line={"shape": "hv"},
)
fig.update_layout(
    template="plotly_white",
    xaxis={"title": "LUTs", "type": "log"},
    yaxis={"title": "throughput (Gbit/s)", "type": "log"},
    legend={"orientation": "h", "y": -0.2},
)
fig.show()

## Read the runs later

The sweep left its runs in the run root. `ws.records` reads them from the disk. It starts no tool
and takes no lock. The filters are `design_name`, `flow` and `target`. A filter that you leave out
matches everything. Failed runs are in the listing too. `print(history)` shows the number of records
and every run directory whose record Xeda could not read. The `netlist` input of a `nextpnr` record
names the run directory of its synthesis, and the record of that run has `nowidelut`.

In [ ]:
history = ws.records(design_name="trivium", flow="nextpnr")
print(history)
widelut = {
    record.run_dir: record.settings["nowidelut"]
    for record in ws.records(design_name="trivium", flow="yosys_fpga")
}

pd.DataFrame(
    [
        {
            "run": record.run_dir.name,
            "outcome": record.outcome.value,
            "bits": record.configuration["rtl"]["parameters"].get("G_OUT_BITS"),
            "nowidelut": widelut.get({i.name: i for i in record.inputs}["netlist"].producer_run_dir),
            "seed": record.settings["seed"],
            "fmax": record.metrics.get("fmax"),
            "lut": record.metrics.get("lut"),
        }
        for record in history
    ]
).sort_values(["bits", "nowidelut", "seed"])

`ws.load_record` reads the record of one run directory. `open` gives the files of the run. While the
`with` block runs, `open` holds a lease on the run directory, so no launch can replace the files.

In [ ]:
record = ws.load_record(sweep.points[0].result.nodes["nextpnr"].run_dir)
print(record.flow, record.outcome.value, record.metric_info["fmax"].unit)
with record.open() as run:
    print(sorted(run.artifacts))
    print(run.output("config").name)
    print(run.artifacts["log"].stat().st_size > 0)

A new run of the same configuration replaces the run directory. A record that you hold from
before does not describe the files any more, and `open` says so. Start the new run outside a
`with record.open()` block: a launch that starts while the thread holds a lease raises
`LeaseHeldError`.

In [ ]:
ws.run(specs[0], rebuild_all=True).require_passed()
try:
    with record.open() as run:
        pass
except RunReplacedError as error:
    print(error.code)
    print(error.expected_run_id != error.found_run_id)

current = ws.load_record(record.run_dir)
print(current.run_id != record.run_id, current.identity == record.identity)

The old record is still valid data, but its files are gone. `ws.load_record` returns the record of the
new run. Both runs have the same configuration, so their `identity` is equal.

## Two cases with nothing to show

A run root with no runs gives an empty listing. It is no error.

In [ ]:
listing = xeda.Workspace(WORK / "empty", local_settings=xeda.local_settings()).records(design_name="trivium")
if not listing.records:
    print("no runs yet: run a sweep first, or open a workspace on a run root that has runs")

A mistyped metric name is an error. `rows` and `to_pandas` raise `KeyError` before they read any
record, and the message lists the metrics that the flow describes.

In [ ]:
name = "nextpnr.fmx"  # a typo for nextpnr.fmax
try:
    sweep.to_pandas(metrics=(name,))
except KeyError as error:
    print(error.args[0])